In [1]:
!pip install pandas pyarrow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

In [2]:
# Load dữ liệu
df = pd.read_parquet('facts.parquet')

# === 1. Thông tin tổng quát ===
print("Shape của dataset:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nInfo:")
print(df.info())
print("\nMô tả thống kê:")
print(df.describe(include='all'))

# Kiểm tra missing values
print("\nMissing values (%):")
print((df.isnull().sum() / len(df) * 100).round(2))

Shape của dataset: (78474497, 5)

Columns: ['adsh', 'tag', 'start', 'end', 'value']

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78474497 entries, 0 to 78474496
Data columns (total 5 columns):
 #   Column  Dtype         
---  ------  -----         
 0   adsh    int64         
 1   tag     category      
 2   start   datetime64[ms]
 3   end     datetime64[ms]
 4   value   float64       
dtypes: category(1), datetime64[ms](2), float64(1), int64(1)
memory usage: 2.5 GB
None

Mô tả thống kê:
                adsh            tag                       start  \
count   7.847450e+07       78474497                    78474497   
unique           NaN           9638                         NaN   
top              NaN  NetIncomeLoss                         NaN   
freq             NaN        1047183                         NaN   
mean    1.187818e+14            NaN  2018-07-23 20:34:59.074000   
min     2.178110e+11            NaN         1900-01-01 00:00:00   
25%     1.000623e+14      

In [4]:
import pandas as pd

# ====================== ĐỌC FILE ======================
df = pd.read_parquet('facts.parquet')

print(f"📊 Original facts.parquet shape: {df.shape}")
print(f"   Số tag unique ban đầu: {df['tag'].nunique():,}")

# ====================== DANH SÁCH TAG CẦN GIỮ ======================
desired_tags = [
    # Biến mục tiêu
    'NetIncomeLoss',
    
    # Nhóm 1 — Doanh thu & Lợi nhuận
    'Revenues',
    'GrossProfit',
    'OperatingIncomeLoss',
    'OperatingExpenses',
    
    # Nhóm 2 — Chi phí
    'GeneralAndAdministrativeExpense',
    'SellingGeneralAndAdministrativeExpense',
    'ShareBasedCompensation',
    'InterestExpense',
    
    # Nhóm 3 — Thuế & Thu nhập trước thuế
    'IncomeTaxExpenseBenefit',
    'IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest',
    'NonoperatingIncomeExpense',
    'OtherNonoperatingIncomeExpense',
    
    # Nhóm 4 — Số cổ phiếu (dùng tính EPS)
    'WeightedAverageNumberOfSharesOutstandingBasic',
    'WeightedAverageNumberOfDilutedSharesOutstanding',
    
    # Nhóm 5 — Bảng cân đối
    'Assets',
    'AssetsCurrent',
    'Liabilities',
    'LiabilitiesCurrent',
    'StockholdersEquity',
    'CashAndCashEquivalentsAtCarryingValue',
    'Goodwill'
]

# ====================== LỌC DỮ LIỆU ======================
extracted = df[df['tag'].isin(desired_tags)].copy()

# Thêm cột thứ tự để sắp xếp theo nhóm bạn liệt kê
order_dict = {tag: i for i, tag in enumerate(desired_tags)}
extracted['order'] = extracted['tag'].map(order_dict)
extracted = extracted.sort_values(by=['adsh', 'order']).drop(columns=['order'])

print(f"\n✅ ĐÃ LỌC XONG!")
print(f"   Số dòng sau khi lọc: {len(extracted):,}")
print(f"   Số tag unique: {extracted['tag'].nunique()} / {len(desired_tags)}")
print(f"   Số adsh unique: {extracted['adsh'].nunique():,}")

# ====================== THỐNG KÊ TẦN SUẤT CÁC TAG ======================
freq = extracted['tag'].value_counts().reset_index()
freq.columns = ['tag', 'frequency']
freq = freq.merge(
    extracted[['tag']].drop_duplicates(), 
    on='tag'
)

print("\n📈 Tần suất từng tag (sau khi lọc):")
print(freq.to_string(index=False))

# ====================== LƯU FILE ======================
extracted.to_parquet('facts_selected_valuation.parquet', index=False)
extracted.to_csv('facts_selected_valuation.csv', index=False)

print("\n💾 Đã lưu 2 file:")
print("   → facts_selected_valuation.parquet  (nhanh nhất cho Python)")
print("   → facts_selected_valuation.csv       (dễ mở Excel)")

# ====================== XEM MẪU 10 DÒNG ĐẦU ======================
print("\n🔍 Sample 10 dòng đầu tiên:")
df.head(10)


📊 Original facts.parquet shape: (78474497, 5)
   Số tag unique ban đầu: 9,638

✅ ĐÃ LỌC XONG!
   Số dòng sau khi lọc: 11,571,711
   Số tag unique: 22 / 22
   Số adsh unique: 273,439

📈 Tần suất từng tag (sau khi lọc):
                                                                                        tag  frequency
                                                                              NetIncomeLoss    1047183
                                                                         StockholdersEquity     870746
                                                                        OperatingIncomeLoss     751511
                                                      CashAndCashEquivalentsAtCarryingValue     724388
                                                                    IncomeTaxExpenseBenefit     674161
                                              WeightedAverageNumberOfSharesOutstandingBasic     653146
                                            WeightedAverageNu

,adsh,tag,start,end,value
0,110465910049632,AccountsPayableCurrent,2010-05-31,2010-05-31,114906000.0
1,110465910063683,AccountsPayableCurrent,2010-05-31,2010-05-31,114906000.0
2,110465911015691,AccountsPayableCurrent,2010-05-31,2010-05-31,114906000.0
3,104746911006302,AccountsPayableCurrent,2010-05-31,2010-05-31,114906000.0
4,110465910049632,AccountsPayableCurrent,2010-08-31,2010-08-31,130112000.0
5,110465910063683,AccountsPayableCurrent,2010-11-30,2010-11-30,129789000.0
6,110465911015691,AccountsPayableCurrent,2011-02-28,2011-02-28,200656000.0
7,104746911006302,AccountsPayableCurrent,2011-05-31,2011-05-31,185096000.0
8,110465911053058,AccountsPayableCurrent,2011-05-31,2011-05-31,185096000.0
9,110465911070822,AccountsPayableCurrent,2011-05-31,2011-05-31,185096000.0


In [5]:
import pandas as pd

# Đọc file và chỉ lấy 3 cột
df = pd.read_parquet('facts_selected_valuation.parquet', 
                     columns=['adsh', 'tag', 'value'])

# Xem kết quả
print(df.head())
print(f'\nKích thước: {df.shape}')

# Lưu ra file mới (nếu cần)
df.to_parquet('facts_3cot.parquet', index=False)
df.to_csv('facts_3cot.csv', index=False)

           adsh            tag         value
0  217811000032  NetIncomeLoss  3.479000e+06
1  217811000032  NetIncomeLoss  1.685000e+06
2  217811000032  NetIncomeLoss  9.172000e+06
3  217811000032  NetIncomeLoss  3.589000e+06
4  217811000032       Revenues  1.080926e+09

Kích thước: (11571711, 3)
